In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

# Initialize and constants
load_dotenv(override=True)

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
#MODEL = 'gpt-5-nano'
 
# ----------Ollama model -----------------------
openai = OpenAI()
api_key = os.getenv('OPENAI_API_KEY')
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key)
MODEL = 'llama3.2:1b'

links = fetch_website_links("https://southwaycoffee.in/")

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Menu page, or Franchise pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "menu page", "url": "https://another.full.url/menu"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

#print(get_links_user_prompt("https://southwaycoffee.in/"))

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )  
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links



API key looks good so far


In [27]:
select_relevant_links("https://southwaycoffee.in/")

Selecting relevant links for https://southwaycoffee.in/ by calling llama3.2:1b
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://southwaycoffee.in/'},
  {'type': 'menu page',
   'url': 'https://www.swiggy.com/menu/1038135?source=sharing'},
  {'type': 'onlineorder', 'url': 'https://southwaycoffee.in/onlineorder/'}]}

In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

print(fetch_page_and_all_relevant_links("https://southwaycoffee.in/"))
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

get_brochure_user_prompt("Southway Coffee", "https://southwaycoffee.in/")

Selecting relevant links for https://southwaycoffee.in/ by calling llama3.2:1b
Found 3 relevant links
## Landing Page:

Southway Coffee | Filter Coffee & Southern Kitchen

×
Home
about us
menu
gallery
contact us
franchise enquiry
Swiggy
Zomato
Order Now
Copyright ©
Genous Indiaahar Private Limited
							All Rights
							Reserved
Best Selling
Southway Coffee
Filter coffee
Sheera Upma combo
⁠Classic Ghee idly
Cheese Cut Masala Dosa
cold coffee
kappiccino
DID YOU KNOW?
ABOUT OUR SERVICE
Famous For Coffee , crispy masala dosa and Southindian
											food
Open Everyday
7:30 AM - 11:00 PM
Located In
Pune
Southway Coffee
Authentic south India cafe , located in near Pimple Saudagar , Rahatni , is
										famous for its strong South Indian coffee, Bangalore-style crispy masala dosa, a
										variety of idlis, and more. Their rich filter coffee pairs perfectly with crispy
										dosa and flavorful chutneys.
Copyright ©
Genous Indiaahar
											Private Limited All Rights Reserved.

"\nYou are looking at a company called: Southway Coffee\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nSouthway Coffee | Filter Coffee & Southern Kitchen\n\n×\nHome\nabout us\nmenu\ngallery\ncontact us\nfranchise enquiry\nSwiggy\nZomato\nOrder Now\nCopyright ©\nGenous Indiaahar Private Limited\r\n\t\t\t\t\t\t\tAll Rights\r\n\t\t\t\t\t\t\tReserved\nBest Selling\nSouthway Coffee\nFilter coffee\nSheera Upma combo\n\u2060Classic Ghee idly\nCheese Cut Masala Dosa\ncold coffee\nkappiccino\nDID YOU KNOW?\nABOUT OUR SERVICE\nFamous For Coffee , crispy masala dosa and Southindian\r\n\t\t\t\t\t\t\t\t\t\t\tfood\nOpen Everyday\n7:30 AM - 11:00 PM\nLocated In\nPune\nSouthway Coffee\nAuthentic south India cafe , located in near Pimple Saudagar , Rahatni , is\r\n\t\t\t\t\t\t\t\t\t\tfamous for its strong South Indian coffee, Bangalore-style crispy masala dosa, 

In [31]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

create_brochure("SouthwayCoffee", "https://southwaycoffee.in/")

Selecting relevant links for https://southwaycoffee.in/ by calling llama3.2:1b
Found 2 relevant links


## Branding and Mission Statement

Southway Coffee is a filter coffee brand dedicated to serving authentic South Indian cuisine, particularly its strong South Indian coffee and crispy masala dosa, alongside a variety of idlis and chutneys.

Our mission is to provide a world-class dining experience that combines the bold flavors of South India with high-quality beverages. We strive to be the go-to destination for those seeking a unique and satisfying meal.

## Customers

- **Foodies**: Our customers are adventurous food enthusiasts who crave authentic South Indian cuisine.
- **Coffee Lovers**: We cater to individuals who appreciate good coffee and enjoy trying new flavors.
- **Family Dinners**: We offer a convenient option for families, with options that can be enjoyed by people of all ages.

## Locations

- **Pune**: Our flagship location is in the heart of Pune, offering a convenient experience for customers across the city.
- **Nearby locations**: Other nearby cafes and eateries have partnered with us to provide customers with access to our services.

## Careers and Job Opportunities

We're always excited to meet passionate individuals who share our vision. Apply now:

- **Frontline Staff**: Join our team as a barista, server, or manager to become part of our company family.
- **Production Team**: Learn the ins and outs of coffee production in our state-of-the-art facility.

## About Us

- **Our Story**: We're more than just a coffee shop – we're a fusion of culture and cuisine. Our story began as a small business, but has grown into a renowned brand with a passion for quality.
- **Meet the Team**: Get to know us better by checking our 'Meet the Team' page.

## Featured Menu Items

Our menu is constantly updated to showcase new flavors and offerings. Check out some of our favorites:

- **Classic Ghee idly**
- **Cheese Cut Masala Dosa**
- **Kappa Cino**

```html
# About Our Service
Our services include:
  Filter Coffee
@ Kappiccino

Southway Coffee | Filter Coffee & Southern Kitchen


# Famous For Coffee , Crispy Masala Dossa and South Indian Food
```

## Terms of Service

### Refunds and Exchanges

*   Offer open from Saturday to Sunday.
*   Offer limited time off during peak hours.

### Payment Gateway
```[1] Paytm Payment Gateway][2]
```
[1] "Paytm Payment Gateway" "Secure Online Payments"
```
Southway Coffee | Filter Coffee & Southern Kitchen